## 1 - Configurações Gerais

In [1]:
#@markdown Detectando o Ambiente de Execução { display-mode: "form" }
try:
    import google.colab
    EXECUTION_ENV = "colab"
except ImportError:
    try:
        import kaggle_secrets
        EXECUTION_ENV = "kaggle"
    except ImportError:
        EXECUTION_ENV = "local"

print(f"AMBIENTE DE EXECUÇÃO DETECTADO: {EXECUTION_ENV}")

AMBIENTE DE EXECUÇÃO DETECTADO: colab


In [2]:
if EXECUTION_ENV in ["colab", "kaggle"]:
    !pip install -q transformers==5.5.4
    !pip install -q minigrid==3.0.0
    !pip install -q langchain_huggingface==1.2.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.7/136.7 kB 7.2 MB/s eta 0:00:00


In [4]:
import os
import sys

if EXECUTION_ENV == "colab":
    repository_path = os.path.join(os.getcwd(), "minigrid_benchmark")

if EXECUTION_ENV == "kaggle":
    repository_path = os.path.join("/kaggle/working/", "minigrid_benchmark")

if EXECUTION_ENV in ["colab", "kaggle"]:
    if not os.path.exists(repository_path):
        !git clone https://github.com/pablo-sampaio/minigrid_benchmark.git
    repository_src_path = os.path.join(repository_path, "src")
    sys.path.append(repository_src_path)
    print(f"Repository src path: {repository_src_path}")

Cloning into 'minigrid_benchmark'...
remote: Enumerating objects: 225, done.
remote: Counting objects: 100% (225/225), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 225 (delta 150), reused 221 (delta 148), pack-reused 0 (from 0)
Receiving objects: 100% (225/225), 427.98 KiB | 5.28 MiB/s, done.
Resolving deltas: 100% (150/150), done.
Repository src path: /content/minigrid_benchmark/src


In [5]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

from react_agent import ReActAgent
from wrappers import SYSTEM_PROMPT_WRAPPER_1, SYSTEM_PROMPT_WRAPPER_2d, OBS_TEMPLATE, OBS_TEMPLATE_ENG

import experiments_util
from experiments_util import run_and_save_experiments, wrapper1, wrapper2_with_numbers, wrapper2_without_numbers

In [6]:
if EXECUTION_ENV == "colab":
    # Mount Google Drive, to save results there
    from google.colab import drive
    drive.mount("/content/drive")
    results_dir = "/content/drive/My Drive/EAD-Pesquisa-Agentes/_projeto_minigrid/results"

if EXECUTION_ENV == "kaggle":
    results_dir = "/kaggle/working/results"

if EXECUTION_ENV in ["colab", "kaggle"]:
    if not os.path.exists(results_dir):
        os.makedirs(results_dir)
    experiments_util.RESULTS_BASE_DIR = results_dir

Mounted at /content/drive


In [7]:
if EXECUTION_ENV == "colab":
    from google.colab import userdata
    if userdata.get("HF_TOKEN"):
        os.environ['HF_TOKEN'] = userdata.get("HF_TOKEN")

if EXECUTION_ENV == "kaggle":
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ['HF_TOKEN'] = user_secrets.get_secret("HF_TOKEN")

if EXECUTION_ENV == "local":
    # just assert that HUGGINGFACE_API_KEY or KF_TOKEN is set
    assert os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_API_KEY"), "Either HUGGINGFACE_API_KEY or HF_TOKEN environment variable must be set"

## 2 - Configurar o Modelo

In [8]:
# Ele efetivamente usa 2 bilhões de parâmetros por vez (note o "2B" no nome), apesar de possuir mais.
# Model card: https://huggingface.co/google/gemma-4-E2B-it
#MODEL_ID = "google/gemma-4-E2B-it"

# Modelos a serem testados
MODEL_ID = "google/gemma-3-1b-it"
#MODEL_ID = "google/gemma-3-4b-it"
#MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
#MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"  # non-thinking version
#MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
#MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
#MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"  # english only!
#MODEL_ID = "microsoft/Phi-3.5-mini-instruct"   # multi-lingual
#MODEL_ID = "HuggingFaceTB/SmolLM3-3B"

In [9]:
local_llm = HuggingFacePipeline.from_model_id(
    model_id=MODEL_ID,
    task="text-generation",
    pipeline_kwargs=dict(
        do_sample=True,
        max_new_tokens=2048,
        return_full_text=False  # Atenção: necessário para contornar um bug!!!
    ),
)

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [10]:
HF_CHAT_MODEL = ChatHuggingFace(llm=local_llm,max_retries=2)

## 3 - Configurações do Experimento

In [11]:
model_name = MODEL_ID.split("/")[-1]
print(f"Model name: {model_name}")

Model name: gemma-3-1b-it


In [12]:
# PARTE 1
configs = [
    {
        'name': f'Prompt_2d_{model_name}_without_history',
        'agent': ReActAgent(HF_CHAT_MODEL, SYSTEM_PROMPT_WRAPPER_2d, OBS_TEMPLATE_ENG, verbose=False),
        'wrapper_fn': wrapper2_with_numbers
    },
    {
        'name': f'Prompt_2d_{model_name}_with_history',
        'agent': ReActAgent(HF_CHAT_MODEL, SYSTEM_PROMPT_WRAPPER_2d, OBS_TEMPLATE_ENG, history_window=5, verbose=False),
        'wrapper_fn': wrapper2_with_numbers
    },
]


In [13]:
# PARTE 2
'''
configs.extend([
    {
        'name': f'Prompt_1_{model_name}_with_history',
        'agent': ReActAgent(HF_CHAT_MODEL, SYSTEM_PROMPT_WRAPPER_1, OBS_TEMPLATE_ENG, history_window=5, verbose=False),
        'wrapper_fn': wrapper1
    },
    {
        'name': f'Prompt_1_{model_name}_without_history',
        'agent': ReActAgent(HF_CHAT_MODEL, SYSTEM_PROMPT_WRAPPER_1, OBS_TEMPLATE_ENG, verbose=False),
        'wrapper_fn': wrapper1
    },
])
#''';

## 4 - Execução do Experimento

In [14]:
# Use um nome novo ou None para iniciar um novo experimento.
# Ou repita um nome já existente na pasta "results" para continuar um experimento.
experiment_name = "Lorraine"

In [15]:
##########################
##  Execute experiments ##
##########################

final_results, filepath = run_and_save_experiments(configs, experiment_name=experiment_name, verbose=True)


Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


KeyboardInterrupt: 

In [ ]:
if EXECUTION_ENV in ["colab", "kaggle"]:
    # zip the final results directory
    import shutil

    experiment_result_dir = os.path.dirname(filepath)
    zip_path = os.path.join(experiments_util.RESULTS_BASE_DIR, f"{experiment_name}_results_zip")

    # Creates results_zip.zip containing everything
    shutil.make_archive(zip_path, 'zip', experiment_result_dir)

    print(f"Created: {zip_path}.zip")

## Plot Results

In [ ]:
print("Now, run notebook \"run_experiments_analysis.ipynb\".")